print(12)

In [5]:
import os
for f in os.listdir("/kaggle/input/datasets/vikramnandimandalam/emotiondetection-preprocessed"):
    print(f)

tokenizer.pickle
padded_sequences.npy
combined_preprocessed.csv


In [6]:
# import numpy as np
# import pandas as pd
# import time
# from sklearn.model_selection import train_test_split
# from sklearn.preprocessing import LabelEncoder
# from sklearn.utils.class_weight import compute_class_weight
# import tensorflow as tf
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout

import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report
import tensorflow as tf

# --- Load preprocessed data ---
combined_df = pd.read_csv("/kaggle/input/datasets/vikramnandimandalam/emotiondetection-preprocessed/combined_preprocessed.csv")
padded_sequences = np.load("/kaggle/input/datasets/vikramnandimandalam/emotiondetection-preprocessed/padded_sequences.npy")


# --- Encode labels ---
label_encoder = LabelEncoder()
labels = label_encoder.fit_transform(combined_df["emotion"])
num_classes = len(label_encoder.classes_)
print("✅ Classes:", list(label_encoder.classes_))

# --- 3-way split: train / val / test ---
X_train, X_temp, y_train, y_temp = train_test_split(
    padded_sequences, labels, test_size=0.3, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print("✅ Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)

# --- Class weights ---
class_weights_array = compute_class_weight(
    class_weight="balanced", classes=np.unique(y_train), y=y_train
)
class_weights = dict(enumerate(class_weights_array))
print("✅ Class weights:", class_weights)


# --- Focal Loss ---
def focal_loss(gamma=2.0):
    def loss_fn(y_true, y_pred):
        y_true_onehot = tf.one_hot(tf.cast(y_true, tf.int32), depth=num_classes)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1 - 1e-7)
        cross_entropy = -y_true_onehot * tf.math.log(y_pred)
        weight = tf.pow(1 - y_pred, gamma)
        loss = weight * cross_entropy
        return tf.reduce_sum(loss, axis=-1)
    return loss_fn

# --- Model (with mask_zero + LSTM dropout) ---
MAX_VOCAB_SIZE = 30000
EMBEDDING_DIM = 128
LSTM_UNITS = 128

model = tf.keras.Sequential([
    # tf.keras.layers.Input(shape=(MAX_SEQ_LEN,)),
    tf.keras.layers.Embedding(
        input_dim=MAX_VOCAB_SIZE,
        output_dim=EMBEDDING_DIM,
        mask_zero=True
    ),
    tf.keras.layers.Bidirectional(
        tf.keras.layers.LSTM(LSTM_UNITS, dropout=0.2, use_cudnn=False)
    ),
    tf.keras.layers.Dense(128, activation="relu"),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3, clipnorm=1.0),
    loss=focal_loss(gamma=2.0),
    metrics=["accuracy"]
)

model.summary()

# --- Callbacks ---
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=2, factor=0.5)
]

# --- Train ---
print("\n🏋️ Training (balanced + focal)...")
start = time.time()

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=10,
    batch_size=512,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)


print(f"\n⏱️ Training time: {(time.time()-start)/60:.2f} min")

# --- Evaluate ---
y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

print("\n📊 CLASSIFICATION REPORT")
print(classification_report(
    y_test,
    y_pred,
    target_names=label_encoder.classes_,
    digits=4
))

print("\nPrediction distribution:")
print(pd.Series(y_pred).value_counts(normalize=True))
print("completed")

✅ Classes: ['Bored', 'Confident', 'Confused', 'Curious', 'Frustrated']
✅ Train: (26361, 80) Val: (5649, 80) Test: (5649, 80)
✅ Class weights: {0: np.float64(2.5105714285714287), 1: np.float64(3.697194950911641), 2: np.float64(1.5250795487416835), 3: np.float64(1.279970866715222), 4: np.float64(0.3455141228127662)}


I0000 00:00:1783859808.479306      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1783859808.482734      58 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


🏋️ Training (balanced + focal)...
Epoch 1/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 29s 339ms/step - accuracy: 0.4603 - loss: 0.6421 - val_accuracy: 0.6568 - val_loss: 0.4005 - learning_rate: 0.0010
Epoch 2/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 17s 321ms/step - accuracy: 0.6922 - loss: 0.2589 - val_accuracy: 0.6968 - val_loss: 0.3299 - learning_rate: 0.0010
Epoch 3/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 17s 324ms/step - accuracy: 0.7964 - loss: 0.1727 - val_accuracy: 0.7361 - val_loss: 0.3177 - learning_rate: 0.0010
Epoch 4/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 17s 322ms/step - accuracy: 0.8444 - loss: 0.1297 - val_accuracy: 0.7456 - val_loss: 0.3232 - learning_rate: 0.0010
Epoch 5/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 16s 317ms/step - accuracy: 0.8698 - loss: 0.1038 - val_accuracy: 0.7437 - val_loss: 0.3602 - learning_rate: 0.0010
Epoch 6/10
52/52 ━━━━━━━━━━━━━━━━━━━━ 17s 330ms/step - accuracy: 0.8942 - loss: 0.0825 - val_accuracy: 0.7483 - val_loss: 0.4166 - learning_rate: 5.0000e-04

⏱️ Training time: 1.88 min

📊 CLASSIFICATI

## # ---Save model ---

In [19]:

import pickle

model.save("/kaggle/working/bilstm_emotion_model.keras")

with open("/kaggle/working/label_encoder.pickle", "wb") as f:
    pickle.dump(label_encoder, f)

print("✅ Model saved to models/bltsm/")

✅ Model saved to models/bltsm/


In [20]:
import tensorflow as tf

model = tf.keras.models.load_model(
    "/kaggle/working/bilstm_emotion_model.keras",  # or wherever you saved it in this session
    compile=False
)
# location /kaggle/working/bilstm_emotion_model.keras

In [21]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('punkt')
nltk.download('stopwords')
english_stopwords = set(stopwords.words('english'))

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\S+", " ", text)
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    tokens = nltk.word_tokenize(text)
    tokens = [t for t in tokens if t not in english_stopwords and len(t) > 1]
    return " ".join(tokens)

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [22]:
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer

combined_df = pd.read_csv("/kaggle/input/datasets/vikramnandimandalam/emotiondetection-preprocessed/combined_preprocessed.csv")

# Drop rows with missing clean_text, and ensure everything is a string
combined_df = combined_df.dropna(subset=["clean_text"])
combined_df["clean_text"] = combined_df["clean_text"].astype(str)

print("✅ Rows after dropping NaN clean_text:", combined_df.shape)

MAX_VOCAB_SIZE = 30000
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(combined_df["clean_text"])

print("✅ Tokenizer rebuilt, vocab size:", len(tokenizer.word_index))

✅ Rows after dropping NaN clean_text: (37557, 3)
✅ Tokenizer rebuilt, vocab size: 18413


In [23]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(combined_df["emotion"])

print("✅ Label encoder rebuilt, classes:", list(label_encoder.classes_))

✅ Label encoder rebuilt, classes: ['Bored', 'Confident', 'Confused', 'Curious', 'Frustrated']


### Step 1: Generate synthetic student-domain data (do this locally — lightweight)
Save as src/generate_student_data.py:

In [26]:
# import pandas as pd

# student_templates = {
#     "Confused": [
#         "I don't understand this formula at all",
#         "Wait, how did we get this answer, I'm lost",
#         "Can you explain that step again, it doesn't make sense",
#         "I have no idea what the professor just said",
#         "This assignment instructions are so unclear",
#         "I keep getting a different answer than the textbook",
#         "None of this is making sense to me right now",
#         "I'm stuck on this problem and don't know where to start",
#     ],
#     "Curious": [
#         "Wait, why does this formula work like that",
#         "I wonder how this concept applies to real projects",
#         "That's interesting, can we go deeper into this topic",
#         "I want to know more about how this algorithm works",
#         "How does this relate to what we learned last week",
#         "This is fascinating, what happens if we change this variable",
#         "I'd like to explore this idea further after class",
#         "What would happen if we applied this to a different problem",
#     ],
#     "Confident": [
#         "I've got this assignment figured out completely",
#         "I understand this topic really well now",
#         "I'm sure I'll do well on this exam",
#         "This makes total sense, I can explain it to others",
#         "I feel ready for the presentation tomorrow",
#         "I solved the whole problem set without help",
#         "I know exactly how to approach this project",
#         "I'm confident I answered all the quiz questions correctly",
#     ],
#     "Frustrated": [
#         "I've tried this problem five times and still can't solve it",
#         "This assignment is taking forever and nothing works",
#         "I'm so annoyed, the code keeps throwing the same error",
#         "Why does this keep failing no matter what I try",
#         "I'm sick of redoing this same section over and over",
#         "This professor's instructions never make sense",
#         "I can't believe I have three assignments due tomorrow",
#         "Nothing about this group project is going smoothly",
#     ],
#     "Bored": [
#         "This lecture is so slow, I already know all of this",
#         "I keep zoning out during this class, nothing is new",
#         "This textbook chapter is putting me to sleep",
#         "Same review material again, I've heard this five times",
#         "I can't focus on this presentation, it's dragging on",
#         "This homework is repetitive and uninteresting",
#         "I'm just waiting for this class to end already",
#         "Nothing about this topic holds my attention today",
#     ],
# }

# fillers = ["", " honestly", " right now", " during class", " today", " again", " seriously"]

# rows = []
# for emotion, templates in student_templates.items():
#     texts = []
#     for t in templates:
#         for f in fillers:
#             texts.append((t + f).strip())
#     target_per_class = 400  # smaller this time — fine-tuning needs less data
#     texts = (texts * ((target_per_class // len(texts)) + 1))[:target_per_class]
#     for txt in texts:
#         rows.append({"text": txt, "emotion": emotion})

# student_df = pd.DataFrame(rows).sample(frac=1, random_state=42).reset_index(drop=True)
# student_df["label"] = student_df["emotion"]  # reuse label2id from BERT setup

# print("✅ Student data:", student_df.shape)
# print(student_df["emotion"].value_counts())

## Step 2: Preprocess using the SAME tokenizer (critical — don't refit)
## Save as src/finetune_preprocess.py:

In [25]:
# import pandas as pd
# import numpy as np
# import pickle
# import re
# import nltk
# from nltk.corpus import stopwords
# from tensorflow.keras.preprocessing.sequence import pad_sequences

# nltk.download('punkt')
# nltk.download('stopwords')
# english_stopwords = set(stopwords.words('english'))

# def clean_text(text):
#     text = str(text).lower()
#     text = re.sub(r"http\S+|www\S+", " ", text)
#     text = re.sub(r"[^a-zA-Z\s]", " ", text)
#     tokens = nltk.word_tokenize(text)
#     tokens = [t for t in tokens if t not in english_stopwords and len(t) > 1]
#     return " ".join(tokens)

# student_df = pd.read_csv("data/student_domain_data.csv")
# student_df["clean_text"] = student_df["text"].apply(clean_text)

# # Load the EXISTING tokenizer — do NOT create a new one
# with open("models/tokenizer.pickle", "rb") as f:
#     tokenizer = pickle.load(f)

# MAX_SEQ_LEN = 80
# sequences = tokenizer.texts_to_sequences(student_df["clean_text"])
# padded = pad_sequences(sequences, maxlen=MAX_SEQ_LEN, padding="post", truncating="post")

# np.save("data/student_padded_sequences.npy", padded)
# student_df.to_csv("data/student_preprocessed.csv", index=False)

# print("✅ Student data preprocessed:", padded.shape)

In [15]:
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(student_df, test_size=0.2, random_state=42, stratify=student_df["label"])

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=80)

train_ds = HFDataset.from_pandas(train_df[["text", "label"]].rename(columns={"label": "labels"}))
val_ds = HFDataset.from_pandas(val_df[["text", "label"]].rename(columns={"label": "labels"}))

train_ds = train_ds.map(tokenize_function, batched=True)
val_ds = val_ds.map(tokenize_function, batched=True)

train_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenized:", len(train_ds), len(val_ds))

Map:   0%|          | 0/1600 [00:00<?, ? examples/s]

TypeError: 'Tokenizer' object is not callable

In [16]:
from torch.utils.data import DataLoader
from torch.optim import AdamW
from tqdm.auto import tqdm

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)

optimizer = AdamW(model.parameters(), lr=1e-5)  # smaller LR than original training (2e-5)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - train"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_losses.append(loss.item())

    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - val"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch["labels"].cpu().numpy())

    from sklearn.metrics import accuracy_score
    val_acc = accuracy_score(val_labels, val_preds)
    print(f"Epoch {epoch+1}: train_loss={sum(train_losses)/len(train_losses):.4f}, val_acc={val_acc:.4f}")

AttributeError: 'Sequential' object has no attribute 'parameters'

In [ ]:
model.save_pretrained("/kaggle/working/bert_student_adaptive")
tokenizer.save_pretrained("/kaggle/working/bert_student_adaptive")
print("✅ Saved: /kaggle/working/bert_student_adaptive")

In [ ]:
test_sentences = [
    "why does this equation flip when i move the variable to the other side",
    "i already know all of this, can we move faster",
    "i totally understand this topic now",
    "why does this keep failing no matter what i try",
]

model.eval()
for s in test_sentences:
    inputs = tokenizer(s, padding="max_length", truncation=True, max_length=80, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)
        pred_id = torch.argmax(probs, dim=-1).item()
        conf = probs[0][pred_id].item()
    print(f"'{s}' -> {id2label[pred_id]} ({conf:.2%})")

In [ ]:
# Step 5: Fine-tune

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
]

print("\n🚀 Fine-tuning on student domain...")
history = adaptive_model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=8,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

In [ ]:
# Step 6: Evaluate + save
import numpy as np
from sklearn.metrics import classification_report

y_pred = np.argmax(adaptive_model.predict(X_val), axis=1)
print("\n📊 Fine-tuned classification report:")
print(classification_report(y_val, y_pred, target_names=label_encoder.classes_))

adaptive_model.save("/kaggle/working/bilstm_student_adaptive.keras")
print("\n✅ Saved: /kaggle/working/bilstm_student_adaptive.keras")

In [ ]:
# Feed the fine-tuned model a genuinely new sentence you just typed yourself — something 
# not derived from any template — and see if it still classifies correctly:

test_sentence = "why does this equation flip when i move the variable to the other side"
cleaned = clean_text(test_sentence)
seq = tokenizer.texts_to_sequences([cleaned])
padded = pad_sequences(seq, maxlen=80, padding="post", truncating="post")
pred = adaptive_model.predict(padded)
predicted_class = label_encoder.classes_[np.argmax(pred)]
print("Predicted:", predicted_class, "| Confidence:", np.max(pred))

In [ ]:
test_sentences = [
    "i keep getting stuck on the same part of this project",
    "this class is going by so fast today, love it",
    "not sure how photosynthesis actually works step by step",
]

for s in test_sentences:
    cleaned = clean_text(s)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=80, padding="post", truncating="post")
    pred = adaptive_model.predict(padded, verbose=0)
    predicted_class = label_encoder.classes_[np.argmax(pred)]
    print(f"'{s}' -> {predicted_class} ({np.max(pred):.2f})")

In [1]:
adaptive_model.save("/kaggle/working/bilstm_student_adaptive.keras")
print("✅ Saved")

# Confirm right after
print(os.path.exists("/kaggle/working/bilstm_student_adaptive.keras"))

NameError: name 'adaptive_model' is not defined

In [4]:
import keras
print(keras.__version__)

3.13.2


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Val")
plt.title("Domain-Adaptive Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("/kaggle/working/domain_adaptive_loss.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
#  accuracy 
plt.figure(figsize=(8, 5))
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Val")
plt.title("Domain-Adaptive Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig("/kaggle/working/domain_adaptive_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()


## BERT Fine-Tuning.

In [ ]:
# BERT Fine-Tuning.
!pip install -q transformers datasets

In [12]:

import pandas as pd
import numpy as np
import torch
from torch.utils.data import DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification
from datasets import Dataset as HFDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("✅ Using device:", device)

✅ Using device: cuda


In [ ]:
combined_df = pd.read_csv("/kaggle/input/datasets/vikramnandimandalam/emotiondetection-preprocessed/combined_preprocessed.csv")
combined_df = combined_df.dropna(subset=["text", "emotion"])

label_encoder = LabelEncoder()
combined_df["label"] = label_encoder.fit_transform(combined_df["emotion"])
num_labels = len(label_encoder.classes_)
combined_df.sample(n=20000, random_state=42)
print("conbined_df")
id2label = {i: c for i, c in enumerate(label_encoder.classes_)}
label2id = {c: i for i, c in enumerate(label_encoder.classes_)}

print("✅ Classes:", id2label)

train_df, temp_df = train_test_split(combined_df, test_size=0.3, random_state=42, stratify=combined_df["label"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df["label"])

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=80)

train_dataset = HFDataset.from_pandas(train_df[["text", "label"]].rename(columns={"label": "labels"}))
val_dataset = HFDataset.from_pandas(val_df[["text", "label"]].rename(columns={"label": "labels"}))
test_dataset = HFDataset.from_pandas(test_df[["text", "label"]].rename(columns={"label": "labels"}))

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("✅ Tokenization complete")

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
).to(device)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print("✅ Model loaded on", device)


In [ ]:
from torch.optim import AdamW
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

optimizer = AdamW(model.parameters(), lr=2e-5)
num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    train_losses = []
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} - train"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        train_losses.append(loss.item())

    model.eval()
    val_losses, val_preds, val_labels = [], [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1} - val"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            val_losses.append(outputs.loss.item())
            preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch["labels"].cpu().numpy())

    val_acc = accuracy_score(val_labels, val_preds)
    val_f1 = f1_score(val_labels, val_preds, average="macro")

    print(f"Epoch {epoch+1}: train_loss={np.mean(train_losses):.4f}, "
          f"val_loss={np.mean(val_losses):.4f}, val_acc={val_acc:.4f}, val_f1_macro={val_f1:.4f}")

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Test"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        labels = batch["labels"].cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels)

from sklearn.metrics import classification_report
print(classification_report(all_labels, all_preds, target_names=list(id2label.values())))

In [ ]:
model.save_pretrained("/kaggle/working/bert_emotion_model_final")
tokenizer.save_pretrained("/kaggle/working/bert_emotion_model_final")

print("✅ Saved full BERT model to /kaggle/working/bert_emotion_model_final")